# Decorator-style and class-style tools

A complete, deterministic course companion.

## Learning objectives

- Attach machine-readable metadata with a decorator.
- Express the same contract as a reusable tool class.
- Verify a deterministic catering quote of 360.

## Setup

Both tool styles are implemented with ordinary Python so their contracts are inspectable offline. The optional cell maps the same idea to `smolagents`.


For the optional live cell, install the dependency with `python -m pip install smolagents`.

## Complete implementation

Run the next cell to build the offline demonstration.

In [1]:
def tool_metadata(*, name, description, inputs, output_type):
    def decorate(function):
        function.tool_name = name
        function.description = description
        function.inputs = inputs
        function.output_type = output_type
        return function
    return decorate

@tool_metadata(
    name="count_place_settings",
    description="Return the number of place settings needed.",
    inputs={"guests": {"type": "integer", "description": "Number of guests."}},
    output_type="integer",
)
def count_place_settings(guests):
    return guests

class CateringQuoteTool:
    name = "catering_quote"
    description = "Multiply the guest count by a per-person price."
    inputs = {
        "guests": {"type": "integer", "description": "Number of guests."},
        "price_per_guest": {"type": "integer", "description": "Price for one guest."},
    }
    output_type = "integer"

    def forward(self, guests, price_per_guest):
        if guests < 1 or price_per_guest < 1:
            raise ValueError("Both inputs must be positive.")
        return guests * price_per_guest

decorated_count = count_place_settings(12)
quote_tool = CateringQuoteTool()
quote_total = quote_tool.forward(guests=decorated_count, price_per_guest=30)

## Deterministic assertions

These checks make the lesson reproducible without a model or network connection.

In [2]:
assert count_place_settings.tool_name == "count_place_settings"
assert count_place_settings.output_type == "integer"
assert quote_tool.name == "catering_quote"
assert set(quote_tool.inputs) == {"guests", "price_per_guest"}
assert quote_total == 360

print("Offline assertions passed.")

Offline assertions passed.


## Results interpretation

The decorator is concise for a function, while the class keeps metadata and execution together for a richer reusable component. Both styles communicate the inputs and output that an agent needs for reliable tool selection.

## Credentials

No credential is used offline. The optional live cell checks `HF_TOKEN` without printing it.

In [3]:
import os

live_model_provider = os.getenv("LIVE_MODEL_PROVIDER", "huggingface").strip().casefold()
hf_token = os.getenv("HF_TOKEN")
if live_model_provider != "local" and not hf_token:
    print("SKIPPED: Set HF_TOKEN to run this optional smolagents example.")
else:
    try:
        from smolagents import Tool, ToolCallingAgent, tool
        if live_model_provider == "local":
            from smolagents import TransformersModel
        else:
            from smolagents import InferenceClientModel
    except ImportError:
        print("SKIPPED: smolagents is not installed; run `python -m pip install smolagents`.")
    else:
        if live_model_provider == "local":
            model = TransformersModel(
                model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct",
                device_map="cpu",
                torch_dtype="auto",
                max_new_tokens=256,
            )
        else:
            model = InferenceClientModel(
                model_id="Qwen/Qwen2.5-Coder-32B-Instruct", token=hf_token, tool_choice="auto"
            )
        @tool
        def live_place_settings(guests: int) -> int:
            """Return a place-setting count.

            Args:
                guests: Positive number of guests.
            """
            return guests

        class LiveCateringQuoteTool(Tool):
            name = "live_catering_quote"
            description = "Return a catering total."
            inputs = {
                "guests": {"type": "integer", "description": "Number of guests."},
                "price_per_guest": {"type": "integer", "description": "Price for one guest."},
            }
            output_type = "integer"

            def forward(self, guests: int, price_per_guest: int) -> int:
                return guests * price_per_guest

        live_agent = ToolCallingAgent(
            tools=[live_place_settings, LiveCateringQuoteTool()], model=model, max_steps=3,
        )
        live_result = live_agent.run("Quote catering for 12 guests at 30 EUR each.")
        print(live_result)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Quote catering for 12 guests at 30 EUR each.                                                                    │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen2.5-Coder-1.5B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'live_catering_quote' with arguments: {'guests': 12, 'price_per_guest': 30}                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 360

... (3 outputs left out)

Observations: 360

Final answer: 360

[Step 2: Duration 43.36 seconds| Input tokens: 2,657 | Output tokens: 71]

360


## Completion checklist

- [ ] Inspected decorator metadata.
- [ ] Instantiated a class-style tool.
- [ ] Verified the quote total.

## Attribution

Original dual-style example informed by the official [smolagents tools guide](https://huggingface.co/docs/smolagents/tutorials/tools) and [tool reference](https://huggingface.co/docs/smolagents/reference/tools). No official notebook is copied. Notebook code is released under the Apache-2.0 metadata license.